In [2]:
import pandas as pd

In [3]:
df = pd.read_csv("../../data/final_train_data/player_data.csv", index_col=False)

In [4]:
df_match = pd. read_csv("../../data/final_train_data/match.csv")

In [5]:
import pandas as pd
import numpy as np

# -----------------------------
# Hyperparameters (easy to tune)
# -----------------------------
ALPHA = 1.3  # weight sharpness

N_SELECT = {
    "FW": 4,
    "MF": 4,
    "DF": 4,
    "GK": 2
}

# -----------------------------
# Selection score
# -----------------------------
df["selection_score"] = (
    0.6 * df["overall_rating"] +
    0.4 * df["base_stats"]
)

# -----------------------------
# Helper: weighted mean
# -----------------------------
def weighted_mean(values, weights):
    return np.sum(values * weights)

# -----------------------------
# Stat formulas (vectorized)
# -----------------------------
def attacking_score(d):
    return (
        0.25 * d["finishing"] +
        0.20 * d["shot_power"] +
        0.20 * d["attack_position"] +
        0.20 * d["ball_control"] +
        0.15 * d["short_passing"]
    )

def defensive_score(d):
    return (
        0.35 * d["defensive_awareness"] +
        0.30 * d["standing_tackle"] +
        0.20 * d["interceptions"] +
        0.15 * d["strength"]
    )

def gk_score(d):
    return (
        0.30 * d["gk_reflexes"] +
        0.25 * d["gk_diving"] +
        0.20 * d["gk_positioning"] +
        0.15 * d["gk_handling"] +
        0.10 * d["reactions"]
    )

def physical_score(d):
    return (
        0.30 * d["sprint_speed"] +
        0.25 * d["acceleration"] +
        0.25 * d["stamina"] +
        0.20 * d["balance"]
    )

# -----------------------------
# Main aggregation
# -----------------------------
team_rows = []

group_keys = ["team", "roster_date"]

for (team, roster_date), g in df.groupby(group_keys):

    selected_players = []

    # ---- player selection per position ----
    for pos, n in N_SELECT.items():
        pos_players = (
            g[g["general_position"] == pos]
            .sort_values("selection_score", ascending=False)
            .head(n)
        )
        if len(pos_players) < n:
            break  # skip incomplete rosters
        selected_players.append(pos_players)

    if len(selected_players) != 4:
        continue

    roster = pd.concat(selected_players)

    # ---- compute position-normalized weights ----
    roster["weight"] = 0.0

    for pos in N_SELECT.keys():
        mask = roster["general_position"] == pos
        scores = roster.loc[mask, "selection_score"] ** ALPHA
        roster.loc[mask, "weight"] = scores / scores.sum()

    # ---- compute individual component scores ----
    roster["A"] = attacking_score(roster)
    roster["D"] = defensive_score(roster)
    roster["G"] = gk_score(roster)
    roster["P"] = physical_score(roster)

    # ---- position-level aggregation ----
    A_FW = weighted_mean(
        roster.loc[roster.general_position == "FW", "A"],
        roster.loc[roster.general_position == "FW", "weight"]
    )
    A_MF = weighted_mean(
        roster.loc[roster.general_position == "MF", "A"],
        roster.loc[roster.general_position == "MF", "weight"]
    )

    D_DF = weighted_mean(
        roster.loc[roster.general_position == "DF", "D"],
        roster.loc[roster.general_position == "DF", "weight"]
    )
    D_MF = weighted_mean(
        roster.loc[roster.general_position == "MF", "D"],
        roster.loc[roster.general_position == "MF", "weight"]
    )

    G_team = weighted_mean(
        roster.loc[roster.general_position == "GK", "G"],
        roster.loc[roster.general_position == "GK", "weight"]
    )

    P_team = weighted_mean(
        roster.loc[roster.general_position != "GK", "P"],
        roster.loc[roster.general_position != "GK", "weight"]
    )

    # ---- team-level scores ----
    A_team = 0.6 * A_FW + 0.4 * A_MF
    D_team = 0.7 * D_DF + 0.3 * D_MF

    T_overall = (
        0.35 * A_team +
        0.30 * D_team +
        0.20 * G_team +
        0.15 * P_team
    )

    team_rows.append({
        "team": team,
        "roster_date": roster_date,
        "attack": A_team,
        "defense": D_team,
        "goalkeeping": G_team,
        "physical": P_team,
        "team_overall": T_overall
    })

# -----------------------------
# Final team-level DataFrame
# -----------------------------
team_stats_df = pd.DataFrame(team_rows)

KeyboardInterrupt: 

In [ ]:
import warnings
warnings.filterwarnings("ignore")


# Import merge module
import sys
sys.path.append('../merge')
from merge import merge_match_with_team_stats

# Merge match data with team statistics
df_match_enriched = merge_match_with_team_stats(df_match, team_stats_df, verbose=True)

# Display sample
print(f"\nSample of enriched data:")
df_match_enriched[['home_team', 'away_team', 'month', 'year', 'home_team_overall', 'away_team_overall', 'home_team_win']].head(10)

In [8]:
df.drop('selection_score', axis=1, inplace=True)

In [9]:
df_sofifa = pd.read_csv("../../data/player_data_train/players_230001.csv")

In [12]:
# max display columns
pd.set_option('display.max_columns', None)
df_sofifa

,Unnamed: 0,Name,Overall rating,Potential,Team & Contract,ID,Birth year,Height,Weight,foot,Best overall,Best position,Growth,Joined,Loan date end,Value,Wage,Release clause,Total attacking,Crossing,Finishing,Heading accuracy,Short passing,Volleys,Total skill,Dribbling,Curve,FK Accuracy,Long passing,Ball control,Total movement,Acceleration,Sprint speed,Agility,Reactions,Balance,Total power,Shot power,Jumping,Stamina,Strength,Long shots,Total mentality,Aggression,Interceptions,Attack position,Vision,Penalties,Composure,Total defending,Defensive awareness,Standing tackle,Sliding tackle,Total goalkeeping,GK Diving,GK Handling,GK Kicking,GK Positioning,GK Reflexes,Total stats,Base stats,Weak foot,Skill moves,Attacking work rate,Defensive work rate,International reputation,Body type,Real face,Pace / Diving,Shooting / Handling,Passing / Kicking,Dribbling / Reflexes,Defending / Pace,Physical / Positioning,Traits,Traits.1,PlayStyles,PlayStyles +,Number of traits,Acceleration type,Club position,Club kit number,Unnamed: 82,roster_date,season_code
0,NaN,K. De BruyneCMCAM,91,91,Manchester City2015 ~ 2025,192985,1991,"181cm 5'11""",70kg 154lbs,Right,91,CM,0,"Aug 30, 2015",NaN,€107.5M,€350K,€198.9M,410,94,85+2,55,93,83,443,88,89+4,83,93,90,394,76,73-3,76-3,91,78,408,92,63,88,74,91,406,75,66,88,94,83,89,186,68,65,53,56,15,13,5,10,13,2303,483,5,4,High,High,4,Unique,Yes,74,88,93,87,64,77,Injury proneLeadershipEarly crosserLong passer...,NaN,NaN,NaN,7,NaN,RCM,17,NaN,"Sep 1, 2022",230001
1,NaN,K. BenzemaCFST,91,91,Real Madrid2009 ~ 2023,165153,1987,"185cm 6'1""",81kg 179lbs,Right,91,CF,0,"Jul 9, 2009",NaN,€64M,€450K,€131.2M,434,75,92,90,89,88,409,87,82,73,76,91,401,79,80,78,92,72,410,87,79,82+1,82,80,367,63,39,92,89,84,90,85,43,24,18,41,13,11,5,5,7,2147,455,4,4,Medium,Medium,4,Normal (170-185),Yes,80,88,83,87,39,78,LeadershipFinesse shotPlaymaker (AI)Outside fo...,NaN,NaN,NaN,5,NaN,CF,9,NaN,"Sep 1, 2022",230001
2,NaN,R. LewandowskiST,91-1,91-1,FC Barcelona2022 ~ 2025,188545,1988,"185cm 6'1""",81kg 179lbs,Right,91,ST,0,"Jul 18, 2022",NaN,€84M,€420K,€172.2M,429,71,94-1,91+1,84-1,89,408,85,79,85,70,89+1,403,76-1,75-3,77,93,82,423,91+1,85,76,87+1,84-3,395,81,49,94-1,81,90,88,96,35,42,19,51,15,6,12,8,10,2205,458,4,4,High,Medium,5,Unique,Yes,75,91,79,86,44,83,Solid playerFinesse shotOutside foot shotChip ...,NaN,NaN,NaN,5,NaN,ST,9,NaN,"Sep 1, 2022",230001
3,NaN,V. van DijkCB,90,90,Liverpool2018 ~ 2025,203376,1991,"193cm 6'4""",92kg 203lbs,Right,90,CB,0,"Jan 1, 2018",NaN,€98M,€230K,€181.3M,316,53,52,87,79,45,362,70,60,70,86,76,362,68-2,91+1,61,89,53,400,81,88,74,93,64,349,85,90,47,65,62,90,270,92,92,86,58,13,10,13,11,11,2117,460,3,2,Medium,High,4,Unique,Yes,81,60,71,72,91,86,LeadershipLong passer (AI)Power header,NaN,NaN,NaN,3,NaN,LCB,4,NaN,"Sep 1, 2022",230001
4,NaN,M. SalahRW,90-1,90-1,Liverpool2017 ~ 2023,209331,1992,"175cm 5'9""",71kg 157lbs,Left,90,RW,0,"Jul 1, 2017",NaN,€115.5M,€270K,€213.7M,400,80-1,93,59,84-1,84,408,90-2,84,69,77,88-2,454,89-1,91,90-1,93-1,91,399,83,69,87,75,85,381,63,55,92,85-1,86,92,122,38,43,41,62,14,14,9,11,14,2226,471,3,4,High,Medium,4,Unique,Yes,90,89,82,90,45,75,Finesse shotLong shot taker (AI)Speed dribbler...,NaN,NaN,NaN,7,NaN,RW,11,NaN,"Sep 1, 2022",230001
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5476,NaN,Wu ChengruST,47,55,Guangzhou City2020 ~ 2022,258965,2000,"178cm 5'10""",71kg 157lbs,Right,50,ST,8,"Sep 1, 2020",NaN,€110K,€1K,€193K,214,41,42,39,51,41,209,45,36,32,48,48,296,53,62,59,49,73,237,52,55,44,48,38,228,51,47,46,48,36,42,125,36,42,47,58,11,10,9,14,14,1367,285,2,2,High,Low,1,Lean (170-185),No,58,43,46,49,41,48,NaN,NaN,NaN,NaN,0,NaN,SUB,28,NaN,"Sep 1, 2022",230001
5477,NaN,Pi ZiyangCM,47,60,Wuhan Yangtze River FC20